# Cloud Provider Analytics — Entrega final
**Big Data 72.80 · ITBA · Arquitectura Lambda · PySpark + Structured Streaming + Cassandra/AstraDB**

Camila Lee (63382) · Lucas Perri (62746)

## 0. Setup — AstraDB y credenciales (una sola vez)
La conexión a AstraDB (Cassandra gestionado) usa un **Secure Connect Bundle (SCB)** + un **application token**.

1. En astra.datastax.com crear una base **Serverless (Non-Vector)** y un keyspace **`cloud_analytics`** desde la UI (en serverless no se puede `CREATE KEYSPACE` desde el driver; sí `CREATE TABLE`).
2. Generar un **application token** (rol *Database Administrator*) → copiar el string `AstraCS:...`.
3. Descargar el **Secure Connect Bundle** (zip) y subirlo a `/content/` (panel **Archivos → Subir**).
4. Subir el dataset para tener `/content/datalake/landing/` con los 7 CSV y `usage_events_stream/*.jsonl`.

### Credenciales por variables de entorno
Las credenciales se leen de **Colab Secrets** o de un archivo **`.env`** (ver `.env.example`). `/content` se borra entre sesiones; para persistir montar Drive.

In [ ]:
# --- 1. Instalación de dependencias ---
!pip -q install pyspark==3.5.1 cassandra-driver python-dotenv
print("instalado")

### 0b. (Opcional) Traer repo + datos automáticamente
Si se corre desde una sesión limpia de Colab, esta celda clona el repo y copia `datalake/landing/` a `/content/`, así no hay necesidad que subir los datos a mano. Si ya se cargaron los datos manualmente, se puede saltear.

*Nota:* las credenciales de AstraDB (token + Secure Connect Bundle) **no** están en el repo (son secretos); esas se cargan aparte en Colab Secrets o `.env`.

In [ ]:
# Clona el repo y deja los datos en /content/datalake/landing/
import os, shutil
REPO_URL = "https://github.com/CamilaBelenLee/cloud_provider_analytics.git"
if not os.path.exists("/content/cloud_provider_analytics"):
    !git clone -q {REPO_URL} /content/cloud_provider_analytics
src = "/content/cloud_provider_analytics/datalake/landing"
dst = "/content/datalake/landing"
if os.path.exists(src) and not os.path.exists(dst):
    os.makedirs("/content/datalake", exist_ok=True)
    shutil.copytree(src, dst)
print("landing presente:", os.path.exists(dst),
      "| archivos:", len(os.listdir(dst)) if os.path.exists(dst) else 0)

In [ ]:
# --- 2. Sesión de Spark ---
# Sesión local: local[*] usa todos los núcleos de la VM de Colab. La misma API/plan
# corre igual en un cluster, así que el código no cambia si el dato crece.
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (StructType, StructField, StringType, DoubleType,
                               LongType, IntegerType, BooleanType, DateType)
spark = (SparkSession.builder.appName("cloud_provider_analytics_mvp")
         .master("local[*]").config("spark.sql.shuffle.partitions","8").getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version)

In [ ]:
# --- 3. Rutas y zonas del Data Lake ---
import os
BASE="/content/datalake"; LANDING=f"{BASE}/landing"
BRONZE=f"{BASE}/bronze"; SILVER=f"{BASE}/silver"; GOLD=f"{BASE}/gold"
QUARANTINE=f"{BASE}/quarantine"; CHK="/content/checkpoints"
for p in [BRONZE,SILVER,GOLD,QUARANTINE,CHK]: os.makedirs(p, exist_ok=True)
EVENTS_DIR=f"{LANDING}/usage_events_stream"
print("landing:", os.path.exists(LANDING), "| events:", os.path.exists(EVENTS_DIR))

## 1. Esquemas explícitos
Usamos schema explícito (no `inferSchema`) porque con estos datos `inferSchema` rompe: `value` viene a veces como número y a veces como string `"100.0"`, y Spark lo leería como double y perdería los strings. El esquema de eventos es uno solo para v1 y v2 — `carbon_kg`/`genai_tokens` quedan nullable, así no hace falta código condicional. `value` se declara **string** y lo casteamos después. Ojo: el campo de tiempo se llama `timestamp`, no `event_time`.

In [ ]:
event_schema = StructType([
    StructField("event_id", StringType()), StructField("timestamp", StringType()),
    StructField("org_id", StringType()), StructField("resource_id", StringType()),
    StructField("service", StringType()), StructField("region", StringType()),
    StructField("metric", StringType()), StructField("value", StringType()),   # string -> cast
    StructField("unit", StringType()), StructField("cost_usd_increment", DoubleType()),
    StructField("schema_version", IntegerType()),
    StructField("carbon_kg", DoubleType()), StructField("genai_tokens", LongType()),  # v2 nullable
])
orgs_schema = StructType([
    StructField("org_id",StringType()),StructField("org_name",StringType()),
    StructField("industry",StringType()),StructField("hq_region",StringType()),
    StructField("plan_tier",StringType()),StructField("is_enterprise",BooleanType()),
    StructField("signup_date",DateType()),StructField("sales_rep",StringType()),
    StructField("lifecycle_stage",StringType()),StructField("marketing_source",StringType()),
    StructField("nps_score",DoubleType())])
users_schema = StructType([
    StructField("user_id",StringType()),StructField("org_id",StringType()),
    StructField("email",StringType()),StructField("role",StringType()),
    StructField("active",BooleanType()),StructField("created_at",DateType()),
    StructField("last_login",DateType())])
billing_schema = StructType([
    StructField("invoice_id",StringType()),StructField("org_id",StringType()),
    StructField("month",DateType()),StructField("subtotal",DoubleType()),
    StructField("credits",DoubleType()),StructField("taxes",DoubleType()),
    StructField("currency",StringType()),StructField("exchange_rate_to_usd",DoubleType())])
tickets_schema = StructType([
    StructField("ticket_id",StringType()),StructField("org_id",StringType()),
    StructField("category",StringType()),StructField("severity",StringType()),
    StructField("created_at",DateType()),StructField("resolved_at",DateType()),
    StructField("csat",DoubleType()),StructField("sla_breached",BooleanType())])
print("esquemas listos")

### 1b. Perfilado de datos (evidencia sobre landing crudo)
Perfilado rápido sobre los datos crudos para ver con números lo que el pipeline tiene que resolver.

In [ ]:
# Perfilado de eventos (lectura batch del landing, sólo para evidencia)
prof = (spark.read.schema(event_schema).json(EVENTS_DIR)
        .withColumn("value_num", F.col("value").cast("double")))
tot = prof.count()
print("eventos totales:", tot, "| event_id únicos:", prof.select("event_id").distinct().count())
prof.groupBy("schema_version").count().orderBy("schema_version").show()
print("value nulos:", prof.filter(F.col("value").isNull()).count())
print("unit nulo con value presente (viola regla 3):",
      prof.filter(F.col("value").isNotNull() & F.col("unit").isNull()).count())
print("cost_usd_increment negativos:", prof.filter(F.col("cost_usd_increment")<0).count())
prof.select(F.round(F.min("cost_usd_increment"),2).alias("min_cost"),
            F.round(F.max("cost_usd_increment"),2).alias("max_cost"),
            F.round(F.expr("percentile_approx(cost_usd_increment,0.99)"),2).alias("p99")).show()
# v1 NO trae la clave carbon_kg -> el esquema explícito la rellena como NULL (debería ser 0 no-nulos en v1)
print("filas v1 con carbon_kg no nulo (esperado 0):",
      prof.filter((F.col("schema_version")==1) & F.col("carbon_kg").isNotNull()).count())

# Perfilado de un par de maestros (issues que NO afectan el mart FinOps, sólo se documentan)
orgs_raw = spark.read.option("header",True).schema(orgs_schema).csv(f"{LANDING}/customers_orgs.csv")
bill_raw = spark.read.option("header",True).schema(billing_schema).csv(f"{LANDING}/billing_monthly.csv")
print("orgs NPS fuera de rango (>100):", orgs_raw.filter(F.col("nps_score")>100).count(),
      "| billing subtotales negativos:", bill_raw.filter(F.col("subtotal")<0).count(),
      "| billing credits vacíos:", bill_raw.filter(F.col("credits").isNull()).count())

# integridad referencial: eventos cuyo org_id no existe en orgs (quedarían con industry/plan nulos tras el join)
org_ids = orgs_raw.select("org_id").distinct()
sin_org = prof.select("org_id").distinct().join(org_ids, on="org_id", how="left_anti").count()
print("org_id de eventos que no existen en orgs (huérfanos):", sin_org)

# conformance: verificamos que los categóricos ya vengan consistentes (sin variantes de casing/espacios)
print("\nvalores distintos de service / region / metric (para decidir si hace falta normalizar):")
prof.select("service").distinct().orderBy("service").show(truncate=False)
prof.select("region").distinct().orderBy("region").show(truncate=False)
prof.select("metric").distinct().orderBy("metric").show(truncate=False)

## 2. Batch a Bronze
Bronze = mismo grano que el CSV, con tipos explícitos, dedupe por clave natural, y `ingest_ts`+`source_file` de procedencia. Parquet particionado.

Particionamos cada maestro por una columna de baja cardinalidad (`hq_region`, `role`, `currency`) para cumplir el requisito de Parquet particionado. En este MVP nadie filtra los maestros por esas columnas, así que es más por el requisito que por una query concreta; en producción se elegiría según los patrones de consulta.

In [ ]:
def csv_a_bronze(path, schema, key, out, part_col=None):
    df = (spark.read.option("header",True).schema(schema).csv(path)
          .withColumn("ingest_ts",F.current_timestamp())
          .withColumn("source_file",F.input_file_name())
          .dropDuplicates([key]))
    w = df.write.mode("overwrite")
    if part_col: w = w.partitionBy(part_col)     # Parquet particionado (requisito 1)
    w.parquet(out); print(f"{out}: {df.count()} filas (particionado por {part_col})"); return df

orgs    = csv_a_bronze(f"{LANDING}/customers_orgs.csv", orgs_schema,   "org_id",    f"{BRONZE}/orgs",    "hq_region")
users   = csv_a_bronze(f"{LANDING}/users.csv",          users_schema,  "user_id",   f"{BRONZE}/users",   "role")
billing = csv_a_bronze(f"{LANDING}/billing_monthly.csv",billing_schema,"invoice_id",f"{BRONZE}/billing", "currency")
tickets = csv_a_bronze(f"{LANDING}/support_tickets.csv",tickets_schema,"ticket_id",f"{BRONZE}/tickets", "category")

## 3. Streaming a Bronze
Leemos la carpeta de JSONLs como stream: dedupe por `event_id`, checkpoint, y `availableNow` porque la fuente es fija (procesa todo y termina; `readStream` no ejecuta nada hasta `writeStream.start()`).

### El watermark: por qué 60 días y no 10 minutos
Los 120 archivos simulan micro-lotes, pero **ninguno es un corte temporal**: cada uno trae eventos de los 60 días completos (`events_part_0000` va del 03-07 al 31-08, igual que el 0119). La fragmentación es aleatoria, no cronológica.

Eso rompe un watermark corto. `dropDuplicates` con watermark descarta lo que llega por debajo de `max(event_time) − delay`; apenas el primer micro-lote ve un evento del 31-08, el watermark salta al final del dataset y **todo julio pasa a estar "atrasado"**. Lo medimos:

| watermark | micro-lotes | eventos en Bronze |
|---|---|---|
| 10 min | 1 (todos los archivos juntos) | 43.200 / 43.200 |
| 10 min | 12 (`maxFilesPerTrigger=10`) | **7.203 / 43.200** |
| 60 días | 12 (`maxFilesPerTrigger=10`) | 43.200 / 43.200 |

Con 10 minutos el pipeline sólo funciona **por casualidad**, porque `availableNow` sin `maxFilesPerTrigger` mete los 120 archivos en un único micro-lote y ahí nada llega tarde. En cuanto Spark parte la corrida en lotes se pierde el 83% de los eventos, sin error ni warning.

Lo ponemos en 60 días, que cubre el span completo del dataset. El estado no se dispara porque la cardinalidad de la clave está acotada (43.200 `event_id` de un histórico cerrado). Con un feed real y llegadas en orden cronológico se dimensionaría al revés: midiendo cuánto llega tarde de verdad y poniendo el watermark ahí.

In [ ]:
# El watermark cubre el span completo del dataset a propósito: los archivos no son cortes
# temporales, cada uno trae los 60 días. Ver la celda de arriba y la prueba de más abajo.
WATERMARK = "60 days"

raw = spark.readStream.schema(event_schema).json(EVENTS_DIR)
bronze_stream = (raw
    .withColumn("event_ts",  F.to_timestamp("timestamp"))
    .withColumn("value_num", F.col("value").cast("double"))   # "100.0"/99.0 -> double; basura -> null
    .withColumn("usage_date",F.to_date("event_ts"))
    .withColumn("ingest_ts", F.current_timestamp())
    .withColumn("source_file", F.input_file_name())
    .withWatermark("event_ts", WATERMARK)
    .dropDuplicates(["event_id"]))
q = (bronze_stream.writeStream.format("parquet")
     .option("path", f"{BRONZE}/events")
     .option("checkpointLocation", f"{CHK}/bronze_events")
     .outputMode("append").trigger(availableNow=True).start())
q.awaitTermination()

n_bronze, n_landing = spark.read.parquet(f"{BRONZE}/events").count(), 43200
print(f"streaming -> bronze: {n_bronze}/{n_landing} eventos",
      "OK" if n_bronze == n_landing else f"FALTAN {n_landing-n_bronze}")

In [ ]:
bronze_events = spark.read.parquet(f"{BRONZE}/events")
print("eventos en bronze:", bronze_events.count())
bronze_events.select("event_id","event_ts","org_id","service","metric","value","value_num",
                     "cost_usd_increment","schema_version","genai_tokens").show(5, truncate=False)

In [ ]:
# Evidencia del watermark: forzamos varios micro-lotes con maxFilesPerTrigger y comparamos.
# Con 10 minutos el dedupe descarta como "tarde" todo lo anterior al máximo ya visto.
import shutil
def probar_watermark(watermark, max_files):
    out, chk = f"/tmp/wm_{watermark.replace(' ','')}_{max_files}", f"/tmp/wmchk_{watermark.replace(' ','')}_{max_files}"
    shutil.rmtree(out, ignore_errors=True); shutil.rmtree(chk, ignore_errors=True)
    df = (spark.readStream.schema(event_schema).option("maxFilesPerTrigger", max_files).json(EVENTS_DIR)
          .withColumn("event_ts", F.to_timestamp("timestamp"))
          .withWatermark("event_ts", watermark)
          .dropDuplicates(["event_id"]))
    qp = (df.writeStream.format("parquet").option("path", out).option("checkpointLocation", chk)
          .outputMode("append").trigger(availableNow=True).start())
    qp.awaitTermination()
    return spark.read.parquet(out).count()

print("los 120 archivos cubren cada uno los 60 días completos, así que 'llegar tarde' acá")
print("no significa nada: depende de en qué micro-lote cae el archivo, no del tiempo del evento.\n")
for wm, mf in [("10 minutes", 10), (WATERMARK, 10)]:
    n = probar_watermark(wm, mf)
    print(f"  watermark={wm:<12} maxFilesPerTrigger={mf}  ->  {n:>6}/43200"
          f"   {'OK' if n==43200 else f'PIERDE {43200-n}'}")

## 4. Silver — calidad + quarantine + enriquecimiento
3 reglas de calidad; las filas que fallan van a **quarantine** (se inspeccionan, no se descartan). Además un `dropDuplicates(["event_id"])` por lotes como red global (la dedupe del stream sólo cubre la ventana del watermark).

Reglas: `event_id` no nulo · `cost_usd_increment ≥ -0.01` · `unit` no nulo cuando `value` existe.

Nota sobre la regla 3: manda a quarantine filas con `unit` nulo aunque tengan un costo válido, así que subestima un poco el costo en Gold. Medimos cuánto cuesta esa decisión abajo. La dejamos así porque la consigna pide esa regla, pero queda documentado el trade-off (en producción se podría conservar el costo y aislar sólo el `value`).

Sobre la dedupe: contamos los `event_id` del landing y son 43.200 únicos sobre 43.200 eventos, o sea que hoy no borra nada. La dejamos igual porque es lo que hace idempotente un re-procesamiento si un archivo se re-entrega, no porque el dataset venga con duplicados.

In [ ]:
ev = spark.read.parquet(f"{BRONZE}/events").dropDuplicates(["event_id"])
valida = (F.col("event_id").isNotNull()
          & (F.col("cost_usd_increment") >= -0.01)
          & ~(F.col("value").isNotNull() & F.col("unit").isNull()))   # regla 3
silver_ok  = ev.filter(valida)
quarantine = ev.filter(~valida)
quarantine.write.mode("overwrite").parquet(f"{QUARANTINE}/events")

# Tickets: csat queda nulo cuando no hubo respuesta (254 de 1000). No lo imputamos con 0.0
# porque hay 11 tickets con csat=0.0 real y quedarían mezclados; avg() ignora los nulos solo.
tickets_bronze = spark.read.parquet(f"{BRONZE}/tickets")
tickets_silver = (tickets_bronze
    .withColumn("ticket_date", F.col("created_at"))
    .withColumn("sla_breached", F.coalesce(F.col("sla_breached"), F.lit(False)))
    .withColumn("abierto", F.col("resolved_at").isNull()))          # 240 tickets sin resolver
# particionamos por severity (4 valores) y no por fecha: son 1000 filas repartidas en 115 días,
# partitionBy(ticket_date) deja 115 parquets de ~8 filas.
tickets_silver.write.mode("overwrite").partitionBy("severity").parquet(f"{SILVER}/tickets")

# Billing: los invoices en USD traen exchange_rate_to_usd ruidoso alrededor de 1 (0.855 a 1.118,
# media 0.9977). Un USD->USD distinto de 1 no existe, así que forzamos 1.0 y dejamos el campo
# para EUR (0.998-1.198) y ARS (0.00133-0.00162), que sí son cotizaciones plausibles.
# En el total global da igual (+0.07%, el ruido es simétrico y se cancela), pero el grano de Q4
# es (org_id, month) y ahí no se cancela: 2.18% de desvío mediano y 17% en el peor caso.
billing_bronze = spark.read.parquet(f"{BRONZE}/billing")
fx = F.when(F.col("currency") == "USD", F.lit(1.0)).otherwise(F.col("exchange_rate_to_usd"))
billing_silver = (billing_bronze
    .withColumn("fx_aplicado", fx)
    .withColumn("subtotal_usd", F.coalesce(F.col("subtotal"), F.lit(0.0)) * fx)
    .withColumn("credits_usd",  F.coalesce(F.col("credits"),  F.lit(0.0)) * fx)   # 137/240 vienen vacíos
    .withColumn("taxes_usd",    F.coalesce(F.col("taxes"),    F.lit(0.0)) * fx)
    .withColumn("net_revenue_usd",
        (F.coalesce(F.col("subtotal"), F.lit(0.0))
         - F.coalesce(F.col("credits"), F.lit(0.0))
         + F.coalesce(F.col("taxes"),   F.lit(0.0))) * fx)
    # 13 subtotales negativos: son notas de crédito, no basura. Las marcamos y las dejamos sumar.
    .withColumn("es_nota_credito", F.col("subtotal") < 0))
billing_silver.write.mode("overwrite").partitionBy("currency").parquet(f"{SILVER}/billing")

print("válidas:", silver_ok.count(), "| en quarantine:", quarantine.count())
# trade-off de la regla 3: costo que se va a quarantine teniendo cost válido
costo_q = quarantine.filter(F.col("cost_usd_increment") >= 0) \
                    .agg(F.round(F.sum("cost_usd_increment"),2).alias("usd")).first()["usd"]
print("costo (USD) en filas quarantineadas con cost válido:", costo_q)
quarantine.select("event_id","value","unit","cost_usd_increment").show(5, truncate=False)

# evidencia del ajuste de FX: cuánto se movería Q4 si multiplicáramos el rate de los USD
_naive = (F.coalesce(F.col("subtotal"),F.lit(0.0)) - F.coalesce(F.col("credits"),F.lit(0.0))
          + F.coalesce(F.col("taxes"),F.lit(0.0))) * F.col("exchange_rate_to_usd")
cmp_fx = (billing_silver.withColumn("net_naive", _naive)
    .groupBy("org_id","month").agg(F.sum("net_revenue_usd").alias("corregido"),
                                   F.sum("net_naive").alias("naive"))
    .withColumn("desvio_pct", F.round(F.abs(F.col("corregido")-F.col("naive"))
                                      / F.abs(F.col("naive")) * 100, 2)))
cmp_fx.select(F.round(F.expr("percentile_approx(desvio_pct,0.5)"),2).alias("desvio_mediano_pct"),
              F.round(F.max("desvio_pct"),2).alias("desvio_max_pct"),
              F.sum(F.when(F.col("desvio_pct")>5,1).otherwise(0)).alias("filas_sobre_5pct"),
              F.count(F.lit(1)).alias("filas_q4")).show()

### Enriquecimiento (broadcast join)
Hacemos `broadcast` de la tabla de orgs (~80 filas) para evitar el shuffle en el join — Spark la manda a cada executor en vez de mover los eventos por la red.

In [ ]:
orgs_dim = spark.read.parquet(f"{BRONZE}/orgs").select(
    "org_id","industry","plan_tier","hq_region","lifecycle_stage")
silver = silver_ok.join(F.broadcast(orgs_dim), on="org_id", how="left")
print("silver enriquecido:", silver.count())

Chequeo del plan: `explain` para confirmar que el join usa `BroadcastHashJoin` (orgs se manda a cada executor, no hay `Exchange`/shuffle por el lado de los eventos).

In [ ]:
# evidencia de que el broadcast funciona: en el plan tiene que aparecer BroadcastHashJoin
silver.explain("formatted")

### 4b. Anomalías de costo — z-score / MAD / p99
Los tres métodos los pide la consigna. El z-score marca de más por la cola de outliers, p99 es medio bruto, MAD aguanta mejor; por eso marcamos sólo cuando coinciden **≥2 de 3** (baja el ruido de ~211 a ~89). Los estadísticos van **por servicio** porque los costos no están en la misma escala. Guardamos qué métodos dispararon en `anomaly_methods` — después lo usamos como `set<text>` en Cassandra. No recortamos el costo: marcamos y seguimos. (`1.4826` lleva el MAD a escala de desvío; `K=1.5` es el factor sobre p99 que elegimos nosotros.)

In [ ]:
stats = silver.groupBy("service").agg(
    F.mean("cost_usd_increment").alias("mu"), F.stddev("cost_usd_increment").alias("sd"),
    F.expr("percentile_approx(cost_usd_increment,0.5)").alias("med"),
    F.expr("percentile_approx(cost_usd_increment,0.99)").alias("p99"))
s1 = silver.join(F.broadcast(stats), on="service", how="left") \
           .withColumn("abs_dev", F.abs(F.col("cost_usd_increment")-F.col("med")))
mad = s1.groupBy("service").agg(F.expr("percentile_approx(abs_dev,0.5)").alias("mad"))
s2  = s1.join(F.broadcast(mad), on="service", how="left")

K = 1.5  # factor sobre p99
flag_z   = (F.abs(F.col("cost_usd_increment")-F.col("mu"))/F.col("sd")) > 3
flag_mad = (F.col("mad")>0) & ((F.col("abs_dev")/(1.4826*F.col("mad"))) > 3)
flag_p99 = F.col("cost_usd_increment") > (F.col("p99")*K)

# guardamos qué métodos dispararon -> después va como set<text> en Cassandra
silver_flagged = (s2
    .withColumn("anomaly_methods", F.array_compact(F.array(
        F.when(flag_z,   F.lit("zscore")),
        F.when(flag_mad, F.lit("mad")),
        F.when(flag_p99, F.lit("p99")))))
    .withColumn("anomaly", F.size("anomaly_methods") >= 2))

silver_flagged.write.mode("overwrite").partitionBy("usage_date","service").parquet(f"{SILVER}/events")
print("anomalías marcadas:", silver_flagged.filter("anomaly").count())
silver_flagged.filter("anomaly").select(
    "event_id","service","cost_usd_increment","anomaly_methods").orderBy(
    F.desc("cost_usd_increment")).show(10, truncate=False)

## 5. Gold — marts de negocio
Cuatro marts, uno por consulta (más el de anomalías que pide la consigna en el punto 4):

| mart | grano | consulta |
|---|---|---|
| `org_daily_usage_by_service` | org × servicio × día | Q1, Q2 |
| `tickets_by_org_date` | org × día | Q3 |
| `revenue_by_org_month` | org × mes | Q4 |
| `genai_tokens_by_org_date` | org × día (sólo `service='genai'`) | Q5 |
| `cost_anomaly_mart` | org × servicio × día | — (requisito 4 de la consigna) |

Features por `metric`, no `count` de filas. Los doubles dejan colas de decimales en el `sum()`, así que redondeamos a centavos.

In [ ]:
se             = spark.read.parquet(f"{SILVER}/events")
silver_tickets = spark.read.parquet(f"{SILVER}/tickets")
silver_billing = spark.read.parquet(f"{SILVER}/billing")

# --- Q1/Q2: FinOps diario -----------------------------------------------------
gold_finops = (se.groupBy("org_id","service","usage_date").agg(
        F.round(F.sum("cost_usd_increment"),2).alias("daily_cost_usd"),
        F.round(F.sum(F.when(F.col("metric")=="requests", F.col("value_num"))),2).alias("requests"),
        F.round(F.sum(F.when(F.col("metric")=="cpu_hours", F.col("value_num"))),4).alias("cpu_hours"),
        F.round(F.sum(F.when(F.col("metric")=="storage_gb_hours", F.col("value_num"))),4).alias("storage_gb_hours"),
        F.sum("genai_tokens").alias("genai_tokens"),
        F.round(F.sum("carbon_kg"),6).alias("carbon_kg"),
        # set de métodos que dispararon ese día (vacío si no hubo anomalía) -> set<text> en Cassandra
        F.array_distinct(F.flatten(F.collect_set("anomaly_methods"))).alias("anomaly_methods"))
    .fillna(0, subset=["requests","cpu_hours","storage_gb_hours","genai_tokens","carbon_kg"]))
gold_finops.write.mode("overwrite").partitionBy("usage_date").parquet(f"{GOLD}/org_daily_usage_by_service")

# --- Q3: soporte --------------------------------------------------------------
# el conteo por severidad va como map<text,int> en Cassandra: son 4 valores, la colección queda acotada
sev = (silver_tickets.groupBy("org_id","ticket_date","severity")
    .agg(F.count(F.lit(1)).alias("n"))
    .groupBy("org_id","ticket_date")
    .agg(F.map_from_entries(F.collect_list(F.struct("severity","n"))).alias("severities")))

gold_tickets = (silver_tickets.groupBy("org_id","ticket_date").agg(
        F.count(F.lit(1)).alias("total_tickets"),
        F.sum(F.when(F.col("severity")=="critical",1).otherwise(0)).alias("critical_tickets"),
        F.sum(F.when(F.col("abierto"),1).otherwise(0)).alias("tickets_abiertos"),
        # avg() saltea los nulos, así que las 254 no-respuestas no bajan el promedio
        F.round(F.avg("csat"),2).alias("avg_csat"),
        F.round(F.avg(F.col("sla_breached").cast("int")),4).alias("sla_breach_rate"))
    .join(sev, on=["org_id","ticket_date"], how="left"))
gold_tickets.write.mode("overwrite").parquet(f"{GOLD}/tickets_by_org_date")

# --- Q4: revenue mensual ------------------------------------------------------
# collect_set de currency porque al agrupar por (org, mes) se pierde en qué moneda facturó
gold_revenue = silver_billing.groupBy("org_id","month").agg(
        F.round(F.sum("subtotal_usd"),2).alias("subtotal_usd"),
        F.round(F.sum("credits_usd"),2).alias("credits_usd"),
        F.round(F.sum("taxes_usd"),2).alias("taxes_usd"),
        F.round(F.sum("net_revenue_usd"),2).alias("net_revenue_usd"),
        F.collect_set("currency").alias("currencies"),
        F.sum(F.when(F.col("es_nota_credito"),1).otherwise(0)).alias("notas_credito"))
gold_revenue.write.mode("overwrite").parquet(f"{GOLD}/revenue_by_org_month")

# --- Q5: tokens GenAI ---------------------------------------------------------
# genai_tokens sólo existe en v2 (desde 2025-07-18) y sólo para service='genai':
# los días de julio antes del corte quedan en 0 tokens con costo > 0, y está bien que así sea.
gold_genai = (se.filter(F.col("service")=="genai")
    .groupBy("org_id","usage_date").agg(
        F.sum("genai_tokens").alias("total_tokens"),
        F.round(F.sum("cost_usd_increment"),4).alias("estimated_cost_usd"))
    .fillna(0, subset=["total_tokens"]))
gold_genai.write.mode("overwrite").parquet(f"{GOLD}/genai_tokens_by_org_date")

# --- mart de anomalías (requisito 4 de la consigna) ---------------------------
gold_anomalias = (se.filter("anomaly").groupBy("org_id","usage_date","service").agg(
        F.count(F.lit(1)).alias("eventos_anomalos"),
        F.round(F.sum("cost_usd_increment"),2).alias("costo_anomalo_usd"),
        F.round(F.max("cost_usd_increment"),2).alias("costo_max_usd"),
        F.array_distinct(F.flatten(F.collect_set("anomaly_methods"))).alias("anomaly_methods")))
gold_anomalias.write.mode("overwrite").parquet(f"{GOLD}/cost_anomaly_mart")

for nombre, df in [("org_daily_usage_by_service", gold_finops), ("tickets_by_org_date", gold_tickets),
                   ("revenue_by_org_month", gold_revenue), ("genai_tokens_by_org_date", gold_genai),
                   ("cost_anomaly_mart", gold_anomalias)]:
    print(f"{nombre:<28} {df.count():>6} filas")
gold_finops.orderBy(F.desc("daily_cost_usd")).show(5, truncate=False)

## 6. Serving — Cassandra/AstraDB
Una tabla por consulta (query-first). La partition key es lo que va con `=` en el `WHERE`; la clustering key ordena y permite el rango.

| tabla | PRIMARY KEY | consulta |
|---|---|---|
| `org_daily_usage_by_service` | `((org_id, service), usage_date)` | Q1 |
| `services_by_org` | `((org_id), service)` | Q2 (índice auxiliar) |
| `tickets_by_org_date` | `((org_id), ticket_date)` | Q3 |
| `revenue_by_org_month` | `((org_id), month)` | Q4 |
| `genai_tokens_by_org_date` | `((org_id), usage_date)` | Q5 |
| `cost_anomaly_mart` | `((org_id), usage_date, service)` | requisito 4 |

Todas con `CLUSTERING ORDER BY (... DESC)`: las consultas miran lo más reciente primero y Cassandra lee secuencial dentro de la partición.

**Colecciones CQL** (las usamos donde el agrupamiento perdería información, no de adorno):
- `anomaly_methods set<text>` — qué métodos dispararon ese día.
- `severities map<text,int>` — conteo por severidad, que se pierde al agrupar por `(org, día)`. Son 4 claves, la colección queda acotada.
- `currencies set<text>` — en qué monedas facturó la org ese mes, que se pierde al normalizar todo a USD.

Credenciales: de **Colab Secrets** o `.env`. `dict_factory` indexa las filas como diccionarios.

In [ ]:
# Carga de credenciales: Colab Secrets -> .env -> os.environ (en ese orden de prioridad)
import os
try:
    from dotenv import load_dotenv; load_dotenv()  # lee un archivo .env si existe
except Exception:
    pass
def get_secret(clave, default=None):
    try:
        from google.colab import userdata
        v = userdata.get(clave)
        if v: return v
    except Exception:
        pass
    return os.getenv(clave, default)

SCB_PATH    = get_secret("SCB_PATH", "/content/secure-connect-YOURDB.zip")
ASTRA_TOKEN = get_secret("ASTRA_TOKEN")            # definir en Secrets/.env (AstraCS:...)
KEYSPACE    = get_secret("ASTRA_KEYSPACE", "cloud_analytics")
assert ASTRA_TOKEN, "Falta ASTRA_TOKEN: cargalo en Colab Secrets o en .env (ver .env.example)"
print("credenciales cargadas | keyspace:", KEYSPACE)

In [ ]:
from cassandra.cluster import Cluster
from cassandra.auth import PlainTextAuthProvider
from cassandra.query import dict_factory
cluster = Cluster(cloud={"secure_connect_bundle": SCB_PATH},
                  auth_provider=PlainTextAuthProvider("token", ASTRA_TOKEN))
session = cluster.connect(KEYSPACE)
session.row_factory = dict_factory
print("conectado:", session.execute("select release_version from system.local").one())

In [ ]:
# Migración de esquema. Ojo: CREATE TABLE IF NOT EXISTS no toca una tabla que ya existe,
# así que si tickets_by_org_date o revenue_by_org_month quedaron creadas por una corrida
# anterior, les faltan las columnas nuevas y el INSERT falla con "Undefined column name".
# ALTER TABLE ADD las agrega sin borrar nada; si ya están, Cassandra tira error y lo ignoramos.
NUEVAS = [
    ("tickets_by_org_date",  "tickets_abiertos bigint"),
    ("tickets_by_org_date",  "severities map<text,int>"),
    ("revenue_by_org_month", "currencies set<text>"),
    ("revenue_by_org_month", "notas_credito int"),
]
for tabla, col in NUEVAS:
    try:
        session.execute(f"ALTER TABLE {KEYSPACE}.{tabla} ADD {col}")
        print(f"  + {tabla}.{col.split()[0]}")
    except Exception as e:
        # tabla inexistente (la crea el CREATE de abajo) o columna ya presente
        print(f"  · {tabla}.{col.split()[0]}: sin cambios ({type(e).__name__})")

In [ ]:
session.execute(f'''
CREATE TABLE IF NOT EXISTS {KEYSPACE}.org_daily_usage_by_service (
  org_id text, service text, usage_date date,
  daily_cost_usd double, requests double, cpu_hours double, storage_gb_hours double,
  genai_tokens bigint, carbon_kg double,
  anomaly_methods set<text>,
  PRIMARY KEY ((org_id, service), usage_date)
) WITH CLUSTERING ORDER BY (usage_date DESC)''')

# índice chico para Q2: "qué servicios tiene una org" en una sola partición (evita escanear la tabla grande)
session.execute(f'''
CREATE TABLE IF NOT EXISTS {KEYSPACE}.services_by_org (
  org_id text, service text,
  PRIMARY KEY ((org_id), service)
)''')

session.execute(f'''
CREATE TABLE IF NOT EXISTS {KEYSPACE}.tickets_by_org_date (
  org_id text, ticket_date date,
  total_tickets bigint, critical_tickets bigint, tickets_abiertos bigint,
  avg_csat double, sla_breach_rate double,
  severities map<text,int>,
  PRIMARY KEY ((org_id), ticket_date)
) WITH CLUSTERING ORDER BY (ticket_date DESC)''')

session.execute(f'''
CREATE TABLE IF NOT EXISTS {KEYSPACE}.revenue_by_org_month (
  org_id text, month date,
  subtotal_usd double, credits_usd double, taxes_usd double, net_revenue_usd double,
  currencies set<text>, notas_credito int,
  PRIMARY KEY ((org_id), month)
) WITH CLUSTERING ORDER BY (month DESC)''')

session.execute(f'''
CREATE TABLE IF NOT EXISTS {KEYSPACE}.genai_tokens_by_org_date (
  org_id text, usage_date date,
  total_tokens bigint, estimated_cost_usd double,
  PRIMARY KEY ((org_id), usage_date)
) WITH CLUSTERING ORDER BY (usage_date DESC)''')

session.execute(f'''
CREATE TABLE IF NOT EXISTS {KEYSPACE}.cost_anomaly_mart (
  org_id text, usage_date date, service text,
  eventos_anomalos int, costo_anomalo_usd double, costo_max_usd double,
  anomaly_methods set<text>,
  PRIMARY KEY ((org_id), usage_date, service)
) WITH CLUSTERING ORDER BY (usage_date DESC, service ASC)''')

print("tablas listas")

In [ ]:
import datetime

# Carga distribuida. Cada partición de Spark abre su propia sesión contra Astra y manda los
# INSERT asíncronos: el mart nunca pasa por el driver. La alternativa (collect / toLocalIterator)
# hace del driver un cuello de botella y no escala más allá de lo que entra en su memoria.
def cargar_a_cassandra(df, cql, a_tupla, lote=500):
    scb, token, ks = SCB_PATH, ASTRA_TOKEN, KEYSPACE

    def por_particion(filas):
        from cassandra.cluster import Cluster
        from cassandra.auth import PlainTextAuthProvider
        cluster = Cluster(cloud={"secure_connect_bundle": scb},
                          auth_provider=PlainTextAuthProvider("token", token))
        ses = cluster.connect(ks)
        try:
            stmt = ses.prepare(cql)          # una sola vez por partición, no por fila
            pend = []
            for r in filas:
                pend.append(ses.execute_async(stmt, a_tupla(r)))
                if len(pend) >= lote:        # ventana acotada: pipeline sin inflar memoria
                    for f in pend: f.result()   # result() propaga el error si alguno falló
                    pend = []
            for f in pend: f.result()
        finally:
            cluster.shutdown()               # sin conexiones colgadas por partición
    df.foreachPartition(por_particion)
    return df.count()

def _fecha(v):
    # el round-trip por parquet puede devolver date, datetime o string
    if isinstance(v, datetime.datetime): return v.date()
    if isinstance(v, str): return datetime.datetime.strptime(v, "%Y-%m-%d").date()
    return v

def _f(v): return float(v) if v is not None else 0.0
def _i(v): return int(v)   if v is not None else 0

CQL_FINOPS = f"""INSERT INTO {KEYSPACE}.org_daily_usage_by_service
 (org_id, service, usage_date, daily_cost_usd, requests, cpu_hours, storage_gb_hours,
  genai_tokens, carbon_kg, anomaly_methods) VALUES (?,?,?,?,?,?,?,?,?,?)"""
CQL_SVC = f"INSERT INTO {KEYSPACE}.services_by_org (org_id, service) VALUES (?,?)"
CQL_TICKETS = f"""INSERT INTO {KEYSPACE}.tickets_by_org_date
 (org_id, ticket_date, total_tickets, critical_tickets, tickets_abiertos, avg_csat,
  sla_breach_rate, severities) VALUES (?,?,?,?,?,?,?,?)"""
CQL_REVENUE = f"""INSERT INTO {KEYSPACE}.revenue_by_org_month
 (org_id, month, subtotal_usd, credits_usd, taxes_usd, net_revenue_usd, currencies,
  notas_credito) VALUES (?,?,?,?,?,?,?,?)"""
CQL_GENAI = f"""INSERT INTO {KEYSPACE}.genai_tokens_by_org_date
 (org_id, usage_date, total_tokens, estimated_cost_usd) VALUES (?,?,?,?)"""
CQL_ANOM = f"""INSERT INTO {KEYSPACE}.cost_anomaly_mart
 (org_id, usage_date, service, eventos_anomalos, costo_anomalo_usd, costo_max_usd,
  anomaly_methods) VALUES (?,?,?,?,?,?,?)"""

def t_finops(r):  return (r["org_id"], r["service"], _fecha(r["usage_date"]),
                          _f(r["daily_cost_usd"]), _f(r["requests"]), _f(r["cpu_hours"]),
                          _f(r["storage_gb_hours"]), _i(r["genai_tokens"]), _f(r["carbon_kg"]),
                          set(r["anomaly_methods"] or []))
def t_svc(r):     return (r["org_id"], r["service"])
# avg_csat va NULL si esa org no tuvo ninguna respuesta ese día: un 0.0 se leería como
# "csat pésimo" en el dashboard en vez de "sin datos"
def t_tickets(r): return (r["org_id"], _fecha(r["ticket_date"]), _i(r["total_tickets"]),
                          _i(r["critical_tickets"]), _i(r["tickets_abiertos"]),
                          float(r["avg_csat"]) if r["avg_csat"] is not None else None,
                          _f(r["sla_breach_rate"]), dict(r["severities"] or {}))
def t_revenue(r): return (r["org_id"], _fecha(r["month"]), _f(r["subtotal_usd"]),
                          _f(r["credits_usd"]), _f(r["taxes_usd"]), _f(r["net_revenue_usd"]),
                          set(r["currencies"] or []), _i(r["notas_credito"]))
def t_genai(r):   return (r["org_id"], _fecha(r["usage_date"]), _i(r["total_tokens"]),
                          _f(r["estimated_cost_usd"]))
def t_anom(r):    return (r["org_id"], _fecha(r["usage_date"]), r["service"],
                          _i(r["eventos_anomalos"]), _f(r["costo_anomalo_usd"]),
                          _f(r["costo_max_usd"]), set(r["anomaly_methods"] or []))

# leemos de Gold en disco (no de los DataFrames en memoria) para que la carga sea
# re-ejecutable sola, que es lo que necesita la prueba de idempotencia de abajo
def cargar_todo():
    g = lambda n: spark.read.parquet(f"{GOLD}/{n}")
    n = {}
    fin = g("org_daily_usage_by_service")
    n["org_daily_usage_by_service"] = cargar_a_cassandra(fin, CQL_FINOPS, t_finops)
    n["services_by_org"]            = cargar_a_cassandra(fin.select("org_id","service").distinct(),
                                                         CQL_SVC, t_svc)
    n["tickets_by_org_date"]        = cargar_a_cassandra(g("tickets_by_org_date"), CQL_TICKETS, t_tickets)
    n["revenue_by_org_month"]       = cargar_a_cassandra(g("revenue_by_org_month"), CQL_REVENUE, t_revenue)
    n["genai_tokens_by_org_date"]   = cargar_a_cassandra(g("genai_tokens_by_org_date"), CQL_GENAI, t_genai)
    n["cost_anomaly_mart"]          = cargar_a_cassandra(g("cost_anomaly_mart"), CQL_ANOM, t_anom)
    return n

for tabla, filas in cargar_todo().items():
    print(f"{tabla:<28} {filas:>6} filas cargadas")

## 7. Las 5 consultas
Los rangos de fechas se derivan **del propio dato** (no del reloj), así la corrida es reproducible. Cada consulta imprime su CQL junto al resultado.

In [ ]:
import pandas as pd, datetime
gold_finops = spark.read.parquet(f"{GOLD}/org_daily_usage_by_service")
muestra = gold_finops.orderBy(F.desc("daily_cost_usd")).first()
ORG, SVC = muestra["org_id"], muestra["service"]
limites = gold_finops.agg(F.min("usage_date").alias("lo"), F.max("usage_date").alias("hi")).first()
LO, HI = limites["lo"], limites["hi"]

# Q1: una sola partición ((org_id, service)) + rango sobre la clustering key
print(f"""-- Q1 · costos y requests diarios por org y servicio en un rango
SELECT usage_date, daily_cost_usd, requests
FROM {KEYSPACE}.org_daily_usage_by_service
WHERE org_id='{ORG}' AND service='{SVC}' AND usage_date>='{LO}' AND usage_date<='{HI}';""")
pd.DataFrame(list(session.execute(
    f"""SELECT usage_date, daily_cost_usd, requests
        FROM {KEYSPACE}.org_daily_usage_by_service
        WHERE org_id=%s AND service=%s AND usage_date>=%s AND usage_date<=%s""",
    (ORG, SVC, LO, HI))))

In [ ]:
# Q2: el top-N es sobre un valor calculado que cruza particiones, así que Cassandra no lo
# resuelve de una. Leemos los servicios de la org desde services_by_org (una partición) y
# sumamos del lado de la app. La alternativa productiva es un mart dedicado top_services_by_org.
corte = HI - datetime.timedelta(days=14)
print(f"""-- Q2 · top-N servicios por costo acumulado, últimos 14 días (org={ORG})
SELECT service FROM {KEYSPACE}.services_by_org WHERE org_id='{ORG}';
SELECT daily_cost_usd FROM {KEYSPACE}.org_daily_usage_by_service
WHERE org_id='{ORG}' AND service=? AND usage_date>='{corte}' AND usage_date<='{HI}';""")

servicios = sorted(r["service"] for r in session.execute(
    f"SELECT service FROM {KEYSPACE}.services_by_org WHERE org_id=%s", (ORG,)))
totales = []
for s in servicios:
    tot = sum((r["daily_cost_usd"] or 0) for r in session.execute(
        f"""SELECT daily_cost_usd FROM {KEYSPACE}.org_daily_usage_by_service
            WHERE org_id=%s AND service=%s AND usage_date>=%s AND usage_date<=%s""",
        (ORG, s, corte, HI)))
    totales.append((s, round(tot, 2)))
pd.DataFrame(sorted(totales, key=lambda x: -x[1]), columns=["service","cost_14d"])

In [ ]:
# Q3: tickets críticos y SLA breach por día. Los tickets arrancan el 2025-05-09, antes que
# los eventos (2025-07-03), así que la ventana de 30 días sale del máximo de tickets.
gold_tickets = spark.read.parquet(f"{GOLD}/tickets_by_org_date")
ORG_T = gold_tickets.orderBy(F.desc("critical_tickets"), F.desc("total_tickets")).first()["org_id"]
HI_T = gold_tickets.agg(F.max("ticket_date").alias("hi")).first()["hi"]
LO_T = HI_T - datetime.timedelta(days=30)

print(f"""-- Q3 · evolución de tickets críticos y tasa de SLA breach por día (últimos 30 días)
SELECT ticket_date, total_tickets, critical_tickets, tickets_abiertos,
       sla_breach_rate, avg_csat, severities
FROM {KEYSPACE}.tickets_by_org_date
WHERE org_id='{ORG_T}' AND ticket_date>='{LO_T}' AND ticket_date<='{HI_T}';""")
pd.DataFrame(list(session.execute(
    f"""SELECT ticket_date, total_tickets, critical_tickets, tickets_abiertos,
               sla_breach_rate, avg_csat, severities
        FROM {KEYSPACE}.tickets_by_org_date
        WHERE org_id=%s AND ticket_date>=%s AND ticket_date<=%s""", (ORG_T, LO_T, HI_T))))

In [ ]:
# Q4: revenue mensual normalizado a USD. currencies muestra en qué monedas facturó la org
# ese mes — se pierde al agrupar, por eso va como set<text>.
gold_revenue = spark.read.parquet(f"{GOLD}/revenue_by_org_month")
ORG_R = gold_revenue.orderBy(F.desc("net_revenue_usd")).first()["org_id"]

print(f"""-- Q4 · revenue mensual con créditos e impuestos, normalizado a USD
SELECT month, subtotal_usd, credits_usd, taxes_usd, net_revenue_usd, currencies, notas_credito
FROM {KEYSPACE}.revenue_by_org_month WHERE org_id='{ORG_R}';""")
pd.DataFrame(list(session.execute(
    f"""SELECT month, subtotal_usd, credits_usd, taxes_usd, net_revenue_usd,
               currencies, notas_credito
        FROM {KEYSPACE}.revenue_by_org_month WHERE org_id=%s""", (ORG_R,))))

In [ ]:
# Q5: tokens GenAI por día. Ojo con la evolución de esquema — genai_tokens aparece recién
# en schema_version=2 (2025-07-18), así que los días de julio previos dan 0 tokens con costo > 0.
gold_genai = spark.read.parquet(f"{GOLD}/genai_tokens_by_org_date")
ORG_G = (gold_genai.groupBy("org_id").agg(F.sum("total_tokens").alias("t"))
         .orderBy(F.desc("t")).first()["org_id"])

print(f"""-- Q5 · tokens GenAI y costo estimado por día
SELECT usage_date, total_tokens, estimated_cost_usd
FROM {KEYSPACE}.genai_tokens_by_org_date WHERE org_id='{ORG_G}';""")
pd.DataFrame(list(session.execute(
    f"""SELECT usage_date, total_tokens, estimated_cost_usd
        FROM {KEYSPACE}.genai_tokens_by_org_date WHERE org_id=%s""", (ORG_G,))))

## 8. Idempotencia
Re-ejecutar la carga no duplica: los archivos son replayables, el stream tiene checkpoint, y la escritura a Cassandra es UPSERT por la primary key de cada tabla. Mostramos el conteo antes y después sobre las 6 tablas — si es idempotente, no cambia ninguno.

In [ ]:
TABLAS = ["org_daily_usage_by_service","services_by_org","tickets_by_org_date",
          "revenue_by_org_month","genai_tokens_by_org_date","cost_anomaly_mart"]
cnt = lambda: {t: session.execute(f"SELECT COUNT(*) AS c FROM {KEYSPACE}.{t}").one()["c"] for t in TABLAS}

antes = cnt()
cargar_todo()          # segunda pasada completa sobre los mismos datos
despues = cnt()

print(f"{'tabla':<28} {'antes':>7} {'después':>8}")
for t in TABLAS:
    print(f"{t:<28} {antes[t]:>7} {despues[t]:>8}   {'OK' if antes[t]==despues[t] else 'DUPLICÓ'}")
print("\nidempotente:", antes == despues)

## 8b. Evidencia de rutas y tamaños (particionado)
Requisito 6: mostramos las rutas de cada zona, su tamaño en disco y las carpetas de partición generadas (`hq_region=`, `usage_date=`, `service=`).

In [ ]:
import subprocess
def evidencia(z):
    du = subprocess.run(["du","-sh",z], capture_output=True, text=True).stdout.strip()
    parts = [p for p in subprocess.run(["ls",z], capture_output=True, text=True).stdout.split()
             if "=" in p]
    print(du)
    if parts: print("   particiones:", parts[:6], ("… (+%d)"%(len(parts)-6)) if len(parts)>6 else "")
for z in [f"{BRONZE}/orgs", f"{BRONZE}/users", f"{BRONZE}/billing", f"{BRONZE}/tickets",
          f"{BRONZE}/events", f"{SILVER}/events", f"{SILVER}/tickets", f"{SILVER}/billing",
          f"{GOLD}/org_daily_usage_by_service", f"{GOLD}/tickets_by_org_date",
          f"{GOLD}/revenue_by_org_month", f"{GOLD}/genai_tokens_by_org_date",
          f"{GOLD}/cost_anomaly_mart"]:
    evidencia(z)

## 9. Speed Layer — streaming agregado → Cassandra con `foreachBatch`
Cassandra no es un sink nativo de Spark, así que usamos `foreachBatch` para mandar UPSERTs por micro-batch. Agregamos por ventana diaria (tumbling) sobre event-time.

Dentro del `foreachBatch` el micro-batch es un DataFrame batch común, así que reusamos el mismo `cargar_a_cassandra` con `foreachPartition`: tampoco acá el dato pasa por el driver.

Aplica las **mismas reglas de calidad que el batch** (incluida la regla 3) y la misma dedupe por `event_id`, para que las dos vías no procesen datos distintos ante las mismas filas sucias. Escribe a `org_daily_usage_stream` (tabla aparte) para no pisar el mart del batch. `outputMode("update")` porque la agregación se actualiza con cada batch.

*Nota sobre el alcance de esta capa*. Con `availableNow=True` sobre los mismos archivos que el batch, esto es una demostración del patrón foreachBatch + ventana + update, no una speed layer completa. En una Lambda real la speed layer correría en paralelo (con `processingTime`), procesando sólo los eventos llegados desde el último batch, y la capa de serving mergearía batch + speed al consultar. Acá la fuente es estática (los JSONL ya están todos), así que no hay "lo nuevo" que una speed layer aportaría: la escribimos a una tabla separada justamente para no mezclar las dos vías. El merge batch+speed en serving queda fuera del alcance.

In [ ]:
# La Speed Layer escribe a SU PROPIA tabla, para no pisar las columnas del mart batch.
session.execute(f'''
CREATE TABLE IF NOT EXISTS {KEYSPACE}.org_daily_usage_stream (
  org_id text, service text, usage_date date,
  daily_cost_usd double, requests double,
  PRIMARY KEY ((org_id, service), usage_date)
) WITH CLUSTERING ORDER BY (usage_date DESC)''')

CQL_STREAM = f"""INSERT INTO {KEYSPACE}.org_daily_usage_stream
 (org_id, service, usage_date, daily_cost_usd, requests) VALUES (?,?,?,?,?)"""

def t_stream(r):
    return (r["org_id"], r["service"], _fecha(r["usage_date"]),
            _f(r["daily_cost_usd"]), _f(r["requests"]))

stream_silver = (spark.readStream.schema(event_schema).json(EVENTS_DIR)
    .withColumn("event_ts", F.to_timestamp("timestamp"))
    .withColumn("value_num", F.col("value").cast("double"))
    # mismas reglas de calidad que el batch (incluida la regla 3) para no procesar datos distintos
    .filter(F.col("event_id").isNotNull()
            & (F.col("cost_usd_increment") >= -0.01)
            & ~(F.col("value").isNotNull() & F.col("unit").isNull()))
    .withWatermark("event_ts", WATERMARK)   # mismo criterio que Bronze
    .dropDuplicates(["event_id"]))            # dedupe igual que el batch

stream_gold = (stream_silver
    .groupBy(F.window("event_ts", "1 day"), "org_id", "service")   # ventana tumbling diaria
    .agg(F.round(F.sum("cost_usd_increment"), 2).alias("daily_cost_usd"),
         F.sum(F.when(F.col("metric")=="requests", F.col("value_num"))).alias("requests"))
    .withColumn("usage_date", F.to_date(F.col("window.start"))))

def a_cassandra(batch_df, batch_id):
    # el micro-batch es un DataFrame batch: va por foreachPartition igual que el mart de Gold
    cargar_a_cassandra(batch_df, CQL_STREAM, t_stream)

qs = (stream_gold.writeStream
      .outputMode("update")
      .foreachBatch(a_cassandra)
      .option("checkpointLocation", f"{CHK}/speed_finops")
      .trigger(availableNow=True)
      .start())
qs.awaitTermination()
n_stream = session.execute(f"SELECT COUNT(*) AS c FROM {KEYSPACE}.org_daily_usage_stream").one()["c"]
print("speed layer OK · filas en org_daily_usage_stream:", n_stream)

## 10. Reporte de evidencia
Corre los criterios de aceptación uno por uno y **escribe cada bloque a `docs/evidence/`** además de imprimirlo. Así los archivos que lee el corrector salen de la misma corrida que produjo los números, sin copiar y pegar a mano.

Si el repo está clonado en `/content/` los archivos van directo a `docs/evidence/`; si no, a `/content/evidence/` para bajarlos y commitearlos.

In [ ]:
# ================================================================
#  REPORTE DE EVIDENCIA — escribe docs/evidence/*.txt y lo imprime
# ================================================================
import io, os, contextlib, subprocess, datetime
import pandas as pd

EVIDENCE = "/content/cloud_provider_analytics/docs/evidence"
if not os.path.isdir(EVIDENCE):
    EVIDENCE = "/content/evidence"
os.makedirs(EVIDENCE, exist_ok=True)

def _emitir(archivo, titulo, fn):
    buf = io.StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            fn()
    except Exception as e:
        print(f"  [falló: {type(e).__name__}: {e}]", file=buf)
    texto = "=" * 66 + f"\n  {titulo}\n" + "=" * 66 + "\n" + buf.getvalue()
    open(f"{EVIDENCE}/{archivo}", "w").write(texto)
    print(texto)

MARTS = ["org_daily_usage_by_service", "tickets_by_org_date", "revenue_by_org_month",
         "genai_tokens_by_org_date", "cost_anomaly_mart"]
TABLAS = MARTS + ["services_by_org"]
n_cas = lambda t: session.execute(f"SELECT COUNT(*) AS c FROM {KEYSPACE}.{t}").one()["c"]

# parámetros derivados del dato, no del reloj
g_fin = spark.read.parquet(f"{GOLD}/org_daily_usage_by_service")
_top  = g_fin.orderBy(F.desc("daily_cost_usd")).first()
ORG, SVC = _top["org_id"], _top["service"]
_b = g_fin.agg(F.min("usage_date").alias("lo"), F.max("usage_date").alias("hi")).first()
LO, HI = _b["lo"], _b["hi"]

def _tabla(filas):
    print(pd.DataFrame(filas).to_string(index=False) if filas else "  (sin filas)")

# ---------------------------------------------------------------- 1
def c1():
    for z, k in [("orgs","batch"),("users","batch"),("billing","batch"),("tickets","batch")]:
        print(f"  Bronze {z:<8} : {spark.read.parquet(f'{BRONZE}/{z}').count():>6} filas  ({k})")
    n = spark.read.parquet(f"{BRONZE}/events").count()
    print(f"  Bronze events   : {n:>6} filas  (streaming)")
    print(f"\n  landing tiene 43200 eventos -> {'sin pérdida' if n==43200 else f'FALTAN {43200-n}'}")
    print(f"  event_id distintos en Bronze: {spark.read.parquet(f'{BRONZE}/events').select('event_id').distinct().count()}")
_emitir("c1.txt", "CRITERIO 1 · Batch y streaming corren con datos provistos", c1)

# ---------------------------------------------------------------- 2
def c2():
    ev = spark.read.parquet(f"{BRONZE}/events").dropDuplicates(["event_id"])
    q  = spark.read.parquet(f"{QUARANTINE}/events")
    val = (F.col("event_id").isNotNull() & (F.col("cost_usd_increment") >= -0.01)
           & ~(F.col("value").isNotNull() & F.col("unit").isNull()))
    v, qn = ev.filter(val).count(), q.count()
    print(f"  Válidas    : {v:>6}")
    print(f"  Quarantine : {qn:>6}  (filas que violan >=1 regla, aisladas)")
    print(f"  Suma       : {v+qn:>6}  = total en Bronze (nada se pierde en silencio)\n")
    print("  Desglose por regla violada (algunas filas violan dos):")
    q.select(F.sum(F.when(F.col("event_id").isNull(),1).otherwise(0)).alias("regla_1_event_id_nulo"),
             F.sum(F.when(F.col("cost_usd_increment") < -0.01,1).otherwise(0)).alias("regla_2_costo_negativo"),
             F.sum(F.when(F.col("value").isNotNull() & F.col("unit").isNull(),1).otherwise(0)).alias("regla_3_unit_nulo")).show()
    print("  Ejemplos de filas aisladas:")
    q.select("event_id","value","unit","cost_usd_increment").show(5, truncate=False)
    c = q.filter(F.col("cost_usd_increment") >= 0).agg(F.round(F.sum("cost_usd_increment"),2)).first()[0]
    print(f"  Trade-off de la regla 3: {c} USD de costo válido queda fuera de Gold.")
_emitir("c2.txt", "CRITERIO 2 · Reglas de calidad y quarantine efectivas", c2)

# ---------------------------------------------------------------- 3
def c3():
    print(f"  {'mart':<30} {'Gold':>7} {'Cassandra':>10}")
    for m in MARTS:
        ng = spark.read.parquet(f"{GOLD}/{m}").count(); nc = n_cas(m)
        print(f"  {m:<30} {ng:>7} {nc:>10}   {'OK' if ng==nc else 'DIFIERE'}")
    print(f"  {'services_by_org (índice)':<30} {'-':>7} {n_cas('services_by_org'):>10}")
    print("\n  Top costo diario:");        g_fin.orderBy(F.desc("daily_cost_usd")).show(5, truncate=False)
    print("  Tickets (colección severities):")
    spark.read.parquet(f"{GOLD}/tickets_by_org_date").orderBy(F.desc("total_tickets")).show(5, truncate=False)
    print("  Revenue (colección currencies):")
    spark.read.parquet(f"{GOLD}/revenue_by_org_month").orderBy(F.desc("net_revenue_usd")).show(5, truncate=False)
_emitir("c3.txt", "CRITERIO 3 · Marts en Gold y filas cargadas en Cassandra", c3)

# ---------------------------------------------------------------- 4 y 5
def c4():
    print(f"""-- Q1 · costos y requests diarios por org y servicio en un rango
SELECT usage_date, daily_cost_usd, requests
FROM {KEYSPACE}.org_daily_usage_by_service
WHERE org_id='{ORG}' AND service='{SVC}' AND usage_date>='{LO}' AND usage_date<='{HI}';
""")
    _tabla(list(session.execute(
        f"""SELECT usage_date, daily_cost_usd, requests FROM {KEYSPACE}.org_daily_usage_by_service
            WHERE org_id=%s AND service=%s AND usage_date>=%s AND usage_date<=%s""", (ORG,SVC,LO,HI))))
_emitir("c4.txt", "CRITERIO 4 · Q1 sobre AstraDB (CQL + resultado)", c4)

def c5():
    corte = HI - datetime.timedelta(days=14)
    print(f"""-- Q2 · top-N servicios por costo acumulado, últimos 14 días (org={ORG})
SELECT service FROM {KEYSPACE}.services_by_org WHERE org_id='{ORG}';
SELECT daily_cost_usd FROM {KEYSPACE}.org_daily_usage_by_service
WHERE org_id='{ORG}' AND service=? AND usage_date>='{corte}' AND usage_date<='{HI}';
""")
    servs = sorted(r["service"] for r in session.execute(
        f"SELECT service FROM {KEYSPACE}.services_by_org WHERE org_id=%s", (ORG,)))
    tot = []
    for s in servs:
        t = sum((r["daily_cost_usd"] or 0) for r in session.execute(
            f"""SELECT daily_cost_usd FROM {KEYSPACE}.org_daily_usage_by_service
                WHERE org_id=%s AND service=%s AND usage_date>=%s AND usage_date<=%s""", (ORG,s,corte,HI)))
        tot.append((s, round(t,2)))
    print(pd.DataFrame(sorted(tot, key=lambda x:-x[1]), columns=["service","cost_14d"]).to_string(index=False))
_emitir("c5.txt", "CRITERIO 5 · Q2 sobre AstraDB (CQL + resultado)", c5)

# ---------------------------------------------------------------- Q3, Q4, Q5
def q3():
    gt = spark.read.parquet(f"{GOLD}/tickets_by_org_date")
    org = gt.orderBy(F.desc("critical_tickets"), F.desc("total_tickets")).first()["org_id"]
    hi  = gt.agg(F.max("ticket_date").alias("hi")).first()["hi"]
    lo  = hi - datetime.timedelta(days=30)
    print(f"""-- Q3 · tickets críticos y tasa de SLA breach por día (últimos 30 días)
SELECT ticket_date, total_tickets, critical_tickets, tickets_abiertos, sla_breach_rate, avg_csat, severities
FROM {KEYSPACE}.tickets_by_org_date
WHERE org_id='{org}' AND ticket_date>='{lo}' AND ticket_date<='{hi}';
""")
    _tabla(list(session.execute(
        f"""SELECT ticket_date, total_tickets, critical_tickets, tickets_abiertos,
                   sla_breach_rate, avg_csat, severities
            FROM {KEYSPACE}.tickets_by_org_date
            WHERE org_id=%s AND ticket_date>=%s AND ticket_date<=%s""", (org, lo, hi))))
    print("\n  avg_csat NULL = esa org no tuvo respuestas de encuesta ese día (no es un 0).")
_emitir("c8_q3.txt", "CRITERIO 6 · Q3 sobre AstraDB (CQL + resultado)", q3)

def q4():
    gr = spark.read.parquet(f"{GOLD}/revenue_by_org_month")
    org = gr.orderBy(F.desc("net_revenue_usd")).first()["org_id"]
    print(f"""-- Q4 · revenue mensual con créditos e impuestos, normalizado a USD
SELECT month, subtotal_usd, credits_usd, taxes_usd, net_revenue_usd, currencies, notas_credito
FROM {KEYSPACE}.revenue_by_org_month WHERE org_id='{org}';
""")
    _tabla(list(session.execute(
        f"""SELECT month, subtotal_usd, credits_usd, taxes_usd, net_revenue_usd, currencies, notas_credito
            FROM {KEYSPACE}.revenue_by_org_month WHERE org_id=%s""", (org,))))
    print("\n  net_revenue_usd = (subtotal - credits + taxes) * fx, con fx=1.0 forzado en USD.")
_emitir("c9_q4.txt", "CRITERIO 7 · Q4 sobre AstraDB (CQL + resultado)", q4)

def q5():
    gg = spark.read.parquet(f"{GOLD}/genai_tokens_by_org_date")
    org = gg.groupBy("org_id").agg(F.sum("total_tokens").alias("t")).orderBy(F.desc("t")).first()["org_id"]
    print(f"""-- Q5 · tokens GenAI y costo estimado por día
SELECT usage_date, total_tokens, estimated_cost_usd
FROM {KEYSPACE}.genai_tokens_by_org_date WHERE org_id='{org}';
""")
    _tabla(list(session.execute(
        f"""SELECT usage_date, total_tokens, estimated_cost_usd
            FROM {KEYSPACE}.genai_tokens_by_org_date WHERE org_id=%s""", (org,))))
    print("\n  Los días previos al 2025-07-18 dan 0 tokens con costo > 0: genai_tokens sólo")
    print("  existe en schema_version=2. Es la evolución de esquema, no un hueco de datos.")
_emitir("c10_q5.txt", "CRITERIO 8 · Q5 sobre AstraDB (CQL + resultado)", q5)

# ---------------------------------------------------------------- idempotencia
def c6():
    antes = {t: n_cas(t) for t in TABLAS}
    cargar_todo()
    despues = {t: n_cas(t) for t in TABLAS}
    print(f"  {'tabla':<30} {'antes':>7} {'después':>8}")
    for t in TABLAS:
        print(f"  {t:<30} {antes[t]:>7} {despues[t]:>8}   {'OK' if antes[t]==despues[t] else 'DUPLICÓ'}")
    print(f"\n  Idempotente: {antes == despues}")
    print("  El INSERT de Cassandra es UPSERT por primary key: re-cargar pisa, no agrega.")
_emitir("c6.txt", "CRITERIO 9 · Reprocesar no duplica (idempotencia)", c6)

# ---------------------------------------------------------------- particionado
def c7():
    for z in [f"{BRONZE}/orgs", f"{BRONZE}/users", f"{BRONZE}/billing", f"{BRONZE}/tickets",
              f"{BRONZE}/events", f"{SILVER}/events", f"{SILVER}/tickets", f"{SILVER}/billing"] + \
             [f"{GOLD}/{m}" for m in MARTS]:
        du = subprocess.run(["du","-sh",z], capture_output=True, text=True).stdout.strip().split("\t")
        parts = [p for p in os.listdir(z) if "=" in p] if os.path.isdir(z) else []
        print(f"  {du[0]:>7}  {z}")
        if parts:
            print(f"           particiones ({len(parts)}): {sorted(parts)[:4]}{' …' if len(parts)>4 else ''}")
_emitir("c7.txt", "CRITERIO + · Rutas y tamaños del particionado", c7)

print(f"\n{'='*66}\n  Evidencia escrita en {EVIDENCE}\n  " +
      ", ".join(sorted(os.listdir(EVIDENCE))) + f"\n{'='*66}")

### 10b. Bajar la evidencia para commitearla
Los `.txt` se escriben adentro de la VM de Colab, que se borra al cerrar la sesión. Esta celda los empaqueta y los baja al navegador para descomprimirlos en `docs/evidence/` del repo local y commitear.

El notebook con sus outputs va por otro lado: **Archivo → Guardar una copia en GitHub**, que commitea el `.ipynb` ejecutado (con resultados) directo a `main`.

In [ ]:
# Empaqueta docs/evidence y lo baja al navegador.
import shutil, os
zip_base = "/content/evidence_export"
shutil.make_archive(zip_base, "zip", EVIDENCE)
print("archivos empaquetados:", sorted(os.listdir(EVIDENCE)))
try:
    from google.colab import files
    files.download(f"{zip_base}.zip")           # descomprimir en docs/evidence/ del repo y commitear
except ImportError:
    print(f"fuera de Colab: el zip quedó en {zip_base}.zip")

## 11. Observabilidad y cierre
Miramos el `lastProgress` de la query de ingesta (cuántas filas entraron, estado del watermark) y cerramos los streams con `stop()`. Dejar streams infinitos colgados en el notebook es problema.

In [ ]:
print("lastProgress (ingesta):", q.lastProgress)
for s in spark.streams.active: s.stop()
print("streams activos:", spark.streams.active)